# # Install dependencies

In [ ]:
!pip install -q --user scikit-learn


# # Load training data

In [ ]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (sklearn / xgboost / lightgbm / keras / skorch)

In [ ]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "train";
#                   "val" enables `t.X_val / t.y_val` + auto-scoring;
#                   "test" populates `t.X_test / t.y_test`)
#   target        — label column name(s); single string or list of strings.
#                   Splits each split's DataFrame into X = features and
#                   y = target so estimators can call `fit(X, y)`.
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `val_accuracy` / `val_r2`. None skips the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
from nubison_model import train
from sklearn.linear_model import LogisticRegression

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    t.fit(LogisticRegression(max_iter=100))

print(f"run_id: {t.run_id}")
